# LLM-as-a-Judge for Reasoning Evaluation
This notebook is a protptye for LLM-as-a-Jduge reasoning evaluation

In [1]:
import tqdm as tqdm

from eval_data import EvalData
from prompt_manager import PromptManager
from vLLM_client import vLLMClient
from vllm import SamplingParams
from typing import Tuple, List, Dict, Any

/home/srahmani/anaconda3/envs/BiasReasoning/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


INFO 09-05 17:41:25 [__init__.py:241] Automatically detected platform cuda.


In [2]:
# Initialize the prompt manager
pm = PromptManager("prompts.json")

In [3]:
# get an instance of EvalData for the "Age" category
eval_data = EvalData(category="Age")

bbq_data = eval_data.load_BBQ_dataset()
bbq_reason_data = eval_data.load_reasoning_data()
templates = eval_data.load_BBQ_templates()

INFO:: Loaded BBQ dataset with 3680 samples.
INFO:: Loaded BBQ reasoning data with 3680 samples.
INFO:: Loaded BBQ templates with 50 samples.


In [4]:
batch_data = []
for index, reasoning_sample in enumerate(bbq_reason_data):
    print(f"Reasoning Sample {index}:")
    batch_data.append({
                'reasoning_trace': reasoning_sample['model_reasoning'],
                'final_answer': reasoning_sample['model_answer']
            })
    if index > 8:
        break

Reasoning Sample 0:
Reasoning Sample 1:
Reasoning Sample 2:
Reasoning Sample 3:
Reasoning Sample 4:
Reasoning Sample 5:
Reasoning Sample 6:
Reasoning Sample 7:
Reasoning Sample 8:
Reasoning Sample 9:


In [5]:
llm_client = vLLMClient()
llm = llm_client.load_vllm()

INFO:: Loading model: Qwen/Qwen3-4B
INFO 09-05 17:42:07 [utils.py:326] non-default args: {'model': 'Qwen/Qwen3-4B', 'trust_remote_code': True, 'max_model_len': 32768, 'enable_prefix_caching': True, 'disable_log_stats': True}


The argument `trust_remote_code` is to be used with Auto classes. It has no effect here and is ignored.


INFO 09-05 17:42:14 [__init__.py:711] Resolved architecture: Qwen3ForCausalLM


`torch_dtype` is deprecated! Use `dtype` instead!


WARNING 09-05 17:42:14 [__init__.py:2768] Your device 'NVIDIA TITAN RTX' (with compute capability 7.5) doesn't support torch.bfloat16. Falling back to torch.float16 for compatibility.
WARNING 09-05 17:42:14 [__init__.py:2819] Casting torch.bfloat16 to torch.float16.
INFO 09-05 17:42:14 [__init__.py:1750] Using max model len 32768
WARNING 09-05 17:42:14 [arg_utils.py:1770] Compute Capability < 8.0 is not supported by the V1 Engine. Falling back to V0. 


...

Loading safetensors checkpoint shards:   0% Completed | 0/3 [00:00<?, ?it/s]
Loading safetensors checkpoint shards:  33% Completed | 1/3 [00:01<00:02,  1.18s/it]
Loading safetensors checkpoint shards:  67% Completed | 2/3 [00:02<00:01,  1.23s/it]
Loading safetensors checkpoint shards: 100% Completed | 3/3 [00:02<00:00,  1.39it/s]
Loading safetensors checkpoint shards: 100% Completed | 3/3 [00:02<00:00,  1.18it/s]


INFO 09-05 17:42:20 [default_loader.py:262] Loading weights took 2.56 seconds


INFO 09-05 17:42:21 [model_runner.py:1112] Model loading took 7.5552 GiB and 3.146062 seconds
INFO 09-05 17:42:25 [worker.py:295] Memory profiling takes 3.77 seconds
INFO 09-05 17:42:25 [worker.py:295] the current vLLM instance can use total_gpu_memory (23.64GiB) x gpu_memory_utilization (0.90) = 21.28GiB
INFO 09-05 17:42:25 [worker.py:295] model weights take 7.56GiB; non_torch_memory takes 0.06GiB; PyTorch activation peak memory takes 2.41GiB; the rest of the memory reserved for KV Cache is 11.25GiB.
INFO 09-05 17:42:25 [executor_base.py:114] # cuda blocks: 5121, # CPU blocks: 1820
INFO 09-05 17:42:25 [executor_base.py:119] Maximum concurrency for 32768 tokens per request: 2.50x
INFO 09-05 17:42:27 [model_runner.py:1383] Capturing cudagraphs for decoding. This may lead to unexpected consequences if the model is not static. To run the model in eager mode, set 'enforce_eager=True' or use '--enforce-eager' in the CLI. If out-of-memory error occurs during cudagraph capture, consider decre

Capturing CUDA graph shapes: 100%|██████████| 35/35 [00:21<00:00,  1.62it/s]

INFO 09-05 17:42:49 [model_runner.py:1535] Graph capturing finished in 22 secs, took 1.29 GiB
INFO 09-05 17:42:49 [llm_engine.py:417] init engine (profile, create kv cache, warmup model) took 27.98 seconds
INFO 09-05 17:42:49 [llm.py:298] Supported_tasks: ['generate']
INFO:: Model loaded successfully: Qwen/Qwen3-4B


In [7]:
# Optimized sampling parameters for Qwen thinking mode (based on official recommendations)
sampling_params = SamplingParams(
    max_tokens=2048,
    temperature=0.6,  # Use user override or default 0.6 for thinking mode
    top_p=0.95,  # Use user override or default 0.95
    top_k=20,  # Use user override or default 20 for thinking mode
    stop=["<|endoftext|>", "<|im_end|>", "<|im_start|>"],  # Qwen specific stop tokens
    skip_special_tokens=False,  # Keep special tokens for proper formatting
    seed=42,  # Set seed for reproducibility
)

In [13]:
messages_batch = [
    [  # Conversation 1
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "Translate 'Hello' to French."},
    ],
    [  # Conversation 2
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "What is 10 * 12?"},
    ],
]

outputs = llm.chat(messages_batch, sampling_params, chat_template_kwargs={"enable_thinking": False}, use_tqdm=False)
for i, output in enumerate(outputs):
    print(f"Conversation {i+1}: {output.outputs[0].text}")

/usr/bin/ld: skipping incompatible /lib/i386-linux-gnu/libcuda.so when searching for -lcuda
/usr/bin/ld: cannot find -lcuda: No such file or directory
/usr/bin/ld: skipping incompatible /lib/i386-linux-gnu/libcuda.so when searching for -lcuda
collect2: error: ld returned 1 exit status


CalledProcessError: Command '['/usr/bin/gcc', '/tmp/tmpeo0ptfbq/main.c', '-O3', '-shared', '-fPIC', '-Wno-psabi', '-o', '/tmp/tmpeo0ptfbq/cuda_utils.cpython-311-x86_64-linux-gnu.so', '-lcuda', '-L/home/srahmani/anaconda3/envs/BiasReasoning/lib/python3.11/site-packages/triton/backends/nvidia/lib', '-L/lib/x86_64-linux-gnu', '-L/lib/i386-linux-gnu', '-I/home/srahmani/anaconda3/envs/BiasReasoning/lib/python3.11/site-packages/triton/backends/nvidia/include', '-I/tmp/tmpeo0ptfbq', '-I/home/srahmani/anaconda3/envs/BiasReasoning/include/python3.11']' returned non-zero exit status 1.

In [14]:
# messages_batch = [
#     [{"role": "user", "content": "What is 2 + 2?"}], 
#     [{"role": "user", "content": "What is 3 + 5?"}]
# ]

prompts = [
    "Hello, my name is",
    "The president of the United States is",
    "The capital of France is",
    "The future of AI is",
]

outputs = llm.chat(
        messages_batch, 
        sampling_params,
        chat_template_kwargs={"enable_thinking": False}, # Disable thinking mode for simplicity
        use_tqdm=False  # Disable vLLM's internal tqdm
)

print(outputs)

/usr/bin/ld: skipping incompatible /lib/i386-linux-gnu/libcuda.so when searching for -lcuda
/usr/bin/ld: cannot find -lcuda: No such file or directory
/usr/bin/ld: skipping incompatible /lib/i386-linux-gnu/libcuda.so when searching for -lcuda
collect2: error: ld returned 1 exit status


CalledProcessError: Command '['/usr/bin/gcc', '/tmp/tmpol_9gga0/main.c', '-O3', '-shared', '-fPIC', '-Wno-psabi', '-o', '/tmp/tmpol_9gga0/cuda_utils.cpython-311-x86_64-linux-gnu.so', '-lcuda', '-L/home/srahmani/anaconda3/envs/BiasReasoning/lib/python3.11/site-packages/triton/backends/nvidia/lib', '-L/lib/x86_64-linux-gnu', '-L/lib/i386-linux-gnu', '-I/home/srahmani/anaconda3/envs/BiasReasoning/lib/python3.11/site-packages/triton/backends/nvidia/include', '-I/tmp/tmpol_9gga0', '-I/home/srahmani/anaconda3/envs/BiasReasoning/include/python3.11']' returned non-zero exit status 1.

In [ ]:
def create_messages(prompt):
    message = [{"role": "user", "content": prompt}]
    return message

In [ ]:
def get_batch_messages(batch_data: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    messages_batch = []
    for item in batch_data:
        prompt = pm.get_prompt("simple_eval", reasoning_trace=item["reasoning_trace"], final_answer=item["final_answer"])
        messages = create_messages(prompt=prompt)
        messages_batch.append(messages)
    return messages_batch

In [ ]:
category = "Age"
batch_size = 1

# Single progress bar for the entire category
for i in range(0, len(batch_data), batch_size):
    batch = batch_data[i:i + batch_size]
    messages_batch = get_batch_messages(batch)
    # Generate outputs using vLLM's chat method with thinking mode
    # Disable internal progress bar by using use_tqdm=False
    # print(messages_batch)
    outputs = llm.chat(
        messages_batch, 
        sampling_params,
        chat_template_kwargs={"enable_thinking": False}, # Disable thinking mode for simplicity
        use_tqdm=False  # Disable vLLM's internal tqdm
    )

    print(outputs)

In [ ]:
# Assuming each row in reasoning_data directly corresponds to the row at the same index in ds
batch_data = []
for index, reasoning_sample in enumerate(bbq_reason_data):
    print(f"Reasoning Sample {index}:")
    print(reasoning_sample)
    print("--------------------------------------------------")
    print(f"Data Sample {index}:")
    ds_sample = bbq_data[index]
    print(ds_sample)
    print("--------------------------------------------------")
    question_index = ds_sample['question_index'] # question_index is str
    template = templates[templates['Q_id'] == int(question_index)] # templates['Q_id'] is int64
    if reasoning_sample['ambiguous']:
        context = template['Ambiguous_Context'].values[0]
        context_type = "ambiguous"
        print(f"Context: {context}")
    elif not reasoning_sample['ambiguous']:
        context = template['Disambiguating_Context'].values[0]
        context_type = "disambiguous"
        print(f"Context: {context}")
    # Answer Options
    ans0, ans1, ans2 = reasoning_sample['ans0'], reasoning_sample['ans1'], reasoning_sample['ans2']
    # Reasoning Trace
    reasoning_trace = reasoning_sample['model_output'] # The model's reasoning process
    # Final Model Answer
    final_answer = reasoning_sample['model_answer'] # The final answer provided by the model
    # Is Correct
    is_correct = reasoning_sample['is_correct'] # a binary value indicating correctness
    break

In [ ]:
# Fill a prompt dynamically
prompt = pm.get_prompt("simple_eval", reasoning_trace="R", final_answer="FA")
print(prompt)